In [ ]:
#!pip install transformers datasets torch scikit-learn

In [ ]:
import pandas as pd
from datasets import Dataset

In [ ]:
# Load CSV into pandas
df = pd.read_csv("huggingdata.csv")

In [ ]:
df.head()

In [ ]:
# Convert to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

In [ ]:
# Split into train and test
dataset = dataset.train_test_split(test_size=0.2)

In [ ]:
# Encode labels
from datasets import ClassLabel

# Collect labels from the full dataset
all_labels = list(dataset['train']['label']) + list(dataset['test']['label'])
unique_labels = sorted(set(all_labels))

class_labels = ClassLabel(num_classes=len(unique_labels), names=unique_labels)

def encode_labels(example):
    # strip spaces, lowercase to be safe
    example['label'] = class_labels.str2int(str(example['label']).strip().lower())
    return example

In [ ]:
dataset = dataset.map(encode_labels)

In [ ]:
#Tokenization
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)
dataset = dataset.map(tokenize, batched=True)
dataset = dataset.remove_columns(["text"]) # keep only input_ids, attention_mask, label
dataset.set_format("torch")

In [ ]:
#Load Model
from transformers import AutoModelForSequenceClassification
num_labels = len(class_labels.names)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

In [ ]:
# Training 
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted")
    }

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()